In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import *

apply_plot_style()
FIGURES_DIR.mkdir(exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = load_raw()
n0 = len(df)
df[TARGET] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])
feats = feature_columns(df)

df[["id", SITE_COL] + feats + [TARGET]].to_csv(
    PROCESSED_DIR / "heart_disease_raw.csv", index=False)

print(f"{n0} records, {len(feats)} features")
print(f"{int((df[TARGET]==0).sum())} negative / {int((df[TARGET]==1).sum())} positive")

In [ ]:
miss = df[feats].isna().mean()
dropped_cols = sorted(miss[miss > MISSING_COL_THRESHOLD].index)
feats = [c for c in feats if c not in dropped_cols]
df = df.drop(columns=dropped_cols)

for c in dropped_cols:
    print(f"dropped {c:<6} {miss[c]*100:.1f}% missing")
print(f"{len(feats)} features left")

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=feats + [TARGET], keep="first").reset_index(drop=True)
print(f"removed {before - len(df)} duplicate records, {len(df)} left")

In [ ]:
CONT = continuous_columns(df, feats)
CATG = [c for c in feats if c not in CONT]

for c in CONT:
    v = df[c].median()
    n = int(df[c].isna().sum())
    df[c] = df[c].fillna(v)
    if n:
        print(f"{c:<9} {n:>3} imputed with median {v:g}")

for c in CATG:
    v = df[c].mode()[0]
    n = int(df[c].isna().sum())
    df[c] = df[c].fillna(v)
    if n:
        print(f"{c:<9} {n:>3} imputed with mode {v:g}")

print(f"\nmissing values remaining: {int(df[feats].isna().sum().sum())}")
print(f"records retained: {len(df)}")
print(df[SITE_COL].value_counts().to_string())

In [ ]:
keep = pd.Series(True, index=df.index)
for c in CONT:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - IQR_MULTIPLIER * iqr, q3 + IQR_MULTIPLIER * iqr
    flagged = (df[c] < lo) | (df[c] > hi)
    print(f"{c:<9} {int(flagged.sum()):>3} outside [{lo:.1f}, {hi:.1f}]")
    keep &= ~flagged

before = len(df)
by_site_before = df[SITE_COL].value_counts()
df = df[keep].reset_index(drop=True)
by_site_after = df[SITE_COL].value_counts()

print(f"\nremoved {before - len(df)} records, {len(df)} left\n")
for s in sorted(by_site_before.index):
    after = by_site_after.get(s, 0)
    flag = "   <-- entire hospital gone" if after == 0 else ""
    print(f"  {s:<12} {by_site_before.get(s,0):>4} -> {after:>4}{flag}")

In [ ]:
levels = {c: sorted(df[c].unique()) for c in CATG}
encoded = pd.get_dummies(df[CATG].astype(int).astype(str), columns=CATG,
                         drop_first=True, dtype=int)

for c in CATG:
    made = [x for x in encoded.columns if x.startswith(c + "_")]
    print(f"{c:<9} {[f'{v:g}' for v in levels[c]]} -> {made}   reference {c}_{levels[c][0]:g}")

df = pd.concat([df[["id", SITE_COL]], df[CONT], encoded, df[[TARGET]]], axis=1)
feats = [c for c in df.columns if c not in NON_FEATURES]
print(f"\n{len(feats)} feature columns")

In [ ]:
const = [c for c in feats if df[c].nunique() <= 1]
if const:
    feats = [c for c in feats if c not in const]
    df = df.drop(columns=const)
print("constant columns dropped:", const or "none")

corr = df[feats].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
redundant = [c for c in upper.columns if (upper[c] > CORRELATION_THRESHOLD).any()]
if redundant:
    feats = [c for c in feats if c not in redundant]
    df = df.drop(columns=redundant)
print("redundant columns dropped:", redundant or "none")
print(f"strongest remaining pair: {upper.max().max():.2f}")

In [ ]:
df[CONT] = (df[CONT] - df[CONT].mean()) / df[CONT].std(ddof=0)

out = df[["id", SITE_COL] + feats + [TARGET]]
out.to_csv(PROCESSED_DIR / "heart_disease_preprocessed.csv", index=False)

print(f"records:  {n0} -> {len(out)}  ({(n0-len(out))/n0*100:.1f}% removed)")
print(f"features: 13 -> {len(feats)}")
print(f"classes:  {int((out[TARGET]==0).sum())} negative / {int((out[TARGET]==1).sum())} "
      f"positive ({out[TARGET].mean()*100:.1f}% positive)")
print(f"hospitals: {', '.join(out[SITE_COL].unique())}")
out.head()

In [ ]:
raw = pd.read_csv(PROCESSED_DIR / "heart_disease_raw.csv")
prep = out
sites = sorted(raw[SITE_COL].unique())

counts = [[int((raw[SITE_COL] == s).sum()) for s in sites],
          [int((prep[SITE_COL] == s).sum()) for s in sites]]
rates = [[frame.loc[frame[SITE_COL] == s, TARGET].mean()
          if (frame[SITE_COL] == s).any() else np.nan for s in sites]
         for frame in (raw, prep)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.6, 3.4))
x = np.arange(len(sites))
names = [f"Before  ({len(raw)} records)", f"After  ({len(prep)} records)"]

for k in (0, 1):
    bars = ax1.bar(x + (k - 0.5) * 0.4, counts[k], 0.36, color=CLASS_COLOURS[k],
                   edgecolor="white", linewidth=1, label=names[k])
    for b, v in zip(bars, counts[k]):
        ax1.text(b.get_x() + b.get_width() / 2, v + 6, str(v), ha="center",
                 fontsize=7.5, color="#52514e")
ax1.set_xticks(x, sites, fontsize=8.5)
ax1.set_ylabel("Records")
ax1.set_title("Records per hospital", fontsize=10, loc="left")
ax1.set_ylim(0, max(counts[0]) * 1.18)
ax1.xaxis.grid(False)
ax1.legend(fontsize=8)

offsets, label_dy = (-0.13, 0.13), (-15, 10)
for k in (0, 1):
    ok = ~np.isnan(rates[k])
    xs, ys = x[ok] + offsets[k], np.array(rates[k])[ok]
    ax2.scatter(xs, ys, s=70, color=CLASS_COLOURS[k], edgecolor="white",
                linewidth=1.2, zorder=3, label=names[k])
    for xi, v in zip(xs, ys):
        ax2.annotate(f"{v:.2f}", (xi, v), textcoords="offset points",
                     xytext=(0, label_dy[k]), ha="center", fontsize=7.5, color="#52514e")
for xi in range(len(sites)):
    if np.isnan(rates[1][xi]):
        ax2.annotate("removed", (xi + offsets[1], rates[0][xi]),
                     textcoords="offset points", xytext=(4, 0), ha="left",
                     va="center", fontsize=7.5, color="#b52d2c")
ax2.set_xticks(x, sites, fontsize=8.5)
ax2.set_ylabel("Positive rate")
ax2.set_ylim(0, 1.12)
ax2.set_title("Disease prevalence per hospital", fontsize=10, loc="left")
ax2.xaxis.grid(False)
ax2.legend(fontsize=8, loc="lower right")

fig.suptitle("Effect of preprocessing on the sample", x=0.02, y=1.04, ha="left", fontsize=11)
fig.savefig(FIGURES_DIR / "eda_before_after.png")
plt.show()